# Can we predict late deliveries?

Goal: predict `late_delivery` from information available **at dispatch**. Plan: inspect data, check for leakage, build features, evaluate with a time-based split against baselines.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv', parse_dates=['dispatched_on'])
print(df.shape, df.consignment.is_unique, df.isna().sum().sum())
df.describe(include='all').T

## Target and leakage check

In [ ]:
print(df.late_delivery.value_counts(normalize=True))
print(df.groupby(df.claim_paid_eur > 0).late_delivery.agg(['mean','size']))

**`claim_paid_eur` is leakage.** It is compensation paid on a claim *settled* for the consignment, so it exists only after delivery, and 90% of consignments with a claim are late (0% of those without). It would make the model look great but is unavailable when a prediction is needed (at dispatch). It is excluded. `consignment` is just an ID and is also excluded.

## Exploration

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.5))
df.groupby('carrier').late_delivery.mean().plot.bar(ax=ax[0], title='Late rate by carrier')
df.groupby(pd.qcut(df.distance_km, 8)).late_delivery.mean().plot.bar(ax=ax[1], title='by distance octile')
df.groupby(pd.qcut(df.weight_kg, 8)).late_delivery.mean().plot.bar(ax=ax[2], title='by weight octile')
df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.mean().plot(ax=ax[3], title='by dispatch month', rot=45)
plt.tight_layout(); plt.show()
print(df.groupby(df.dispatched_on.dt.hour // 4 * 4).late_delivery.mean().round(3).to_string())
print(df.groupby(df.dispatched_on.dt.dayofweek).late_delivery.mean().round(3).to_string())

## The late rate drifts strongly over time
The monthly late rate swings between ~1% and ~49% and collapses after Sept 2025. Whatever drives this is not in the data, so absolute probabilities will not transfer across periods; we judge the model mainly on *ranking* (AUC) and are careful with the split.

In [ ]:
m = df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.mean().round(3)
m.plot(kind='bar', figsize=(12,3), title='Late rate by dispatch month'); plt.show()
print(df.groupby([df.dispatched_on.dt.to_period('Q'), 'carrier']).late_delivery.mean().unstack().round(3))

## Features and time-based split
Dispatch time becomes hour/weekday/weekend features (calendar month is deliberately not used: the late rate drifts over time, so month would not generalise). We train on the earlier 75% of consignments and test on the most recent 25%, which mimics predicting future shipments and avoids optimistic random-split estimates.

In [ ]:
d = df.sort_values('dispatched_on').reset_index(drop=True)
d['hour'] = d.dispatched_on.dt.hour
d['dow'] = d.dispatched_on.dt.dayofweek
d['weekend'] = (d.dow >= 5).astype(int)
d['kg_per_parcel'] = d.weight_kg / d.parcels
num = ['distance_km','weight_kg','parcels','kg_per_parcel','hour','dow','weekend']
cat = ['carrier']
cut = int(len(d) * 0.75)
train, test = d.iloc[:cut], d.iloc[cut:]
Xtr, ytr, Xte, yte = train[num+cat], train.late_delivery, test[num+cat], test.late_delivery
print(train.dispatched_on.max(), test.dispatched_on.min(), ytr.mean().round(3), yte.mean().round(3))

## Models
Baselines: constant (base rate) and carrier-only rate. Then logistic regression and gradient boosting.

In [ ]:
pre = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(), cat)])
models = {
 'logreg': make_pipeline(pre, LogisticRegression(max_iter=1000)),
 'hist_gb': make_pipeline(ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(),cat)]),
                          HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, random_state=0)),
}
carrier_rate = ytr.groupby(Xtr.carrier).mean()
cp = Xte.carrier.map(carrier_rate)
rows = [('base rate', 0.5, yte.mean()), ('carrier-only', roc_auc_score(yte, cp), average_precision_score(yte, cp))]
probs = {}
for n, m in models.items():
    m.fit(Xtr, ytr); probs[n] = m.predict_proba(Xte)[:,1]
    rows.append((n, roc_auc_score(yte, probs[n]), average_precision_score(yte, probs[n])))
pd.DataFrame(rows, columns=['model','ROC AUC','PR AUC (base rate = %.3f)' % yte.mean()]).set_index('model').round(3)

In [ ]:
rng = np.random.default_rng(0); p = probs['logreg']; y = yte.values
aucs = [roc_auc_score(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(500))]
print('logreg AUC 95%% CI: %.3f-%.3f' % tuple(np.percentile(aucs, [2.5, 97.5])))
RocCurveDisplay.from_predictions(y, p, name='logreg'); RocCurveDisplay.from_predictions(y, probs['hist_gb'], name='hist_gb', ax=plt.gca())
plt.plot([0,1],[0,1],'k--'); plt.show()

## What drives lateness?

In [ ]:
pi = permutation_importance(models['hist_gb'], Xte, yte, scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, index=num+cat).sort_values().plot.barh(title='Permutation importance (test AUC drop)'); plt.show()

## Thresholds and calibration

In [ ]:
p = probs['logreg']
for t in [0.2, 0.3, 0.4, 0.5]:
    pred = p >= t
    print(f't={t}: flagged {pred.mean():.1%}, precision {yte[pred].mean():.3f}, recall {pred[yte==1].mean():.3f}')
cal = pd.DataFrame({'p': p, 'y': yte.values}).groupby(pd.qcut(p, 10, duplicates='drop')).mean()
cal.plot(x='p', y='y', marker='o', title='Calibration (deciles)'); plt.plot([0,1],[0,1],'k--'); plt.show()

## Rolling-origin evaluation and within-period ranking
The single 75/25 split lands almost entirely in the low-late regime (test base rate ~5%), a harsh and atypical test. A fairer view: repeatedly train on everything before a cutoff and test on the next 3 months, and also measure AUC within each month (which removes the base-rate shift).

In [ ]:
t0 = d.dispatched_on.min(); feats = num + cat
rows = []
for a in [12, 15, 18, 21]:
    s, e = t0 + pd.DateOffset(months=a), t0 + pd.DateOffset(months=a+3)
    tr, te = d[d.dispatched_on < s], d[(d.dispatched_on >= s) & (d.dispatched_on < e)]
    r = {'test window': f'{s:%Y-%m}..{e:%Y-%m}', 'late rate': te.late_delivery.mean()}
    for n, m in models.items():
        r[n] = roc_auc_score(te.late_delivery, m.fit(tr[feats], tr.late_delivery).predict_proba(te[feats])[:,1])
    rows.append(r)
print(pd.DataFrame(rows).round(3).to_string(index=False))
# refit on the original split, then AUC within each month of the test period
for n, m in models.items(): m.fit(Xtr, ytr)
t = test.assign(p=models['logreg'].predict_proba(Xte)[:,1], month=test.dispatched_on.dt.to_period('M'))
print(t.groupby('month')[['late_delivery','p']].apply(lambda g: pd.Series({'n': len(g), 'late rate': g.late_delivery.mean(), 'AUC': roc_auc_score(g.late_delivery, g.p) if g.late_delivery.nunique() > 1 else np.nan})).round(3))

## Conclusions

* **Partly yes.** Distance, weight and carrier carry a real ranking signal (see the rolling-origin AUCs above, versus 0.5 for chance). Longer and heavier consignments are later; `coastal` is the worst carrier. Hour, weekday and parcel count add essentially nothing.
* **`claim_paid_eur` must not be used.** It looks almost perfectly predictive but is only known after delivery.
* **Absolute probabilities are unreliable.** The late rate swings from ~1% to ~49% by month and fell sharply from Oct 2025, for reasons not in the data (network disruption, a service change, or a change in how lateness is recorded are all possible; worth asking the business). A model trained on earlier data over-predicts on the recent regime, and the fixed 75/25 split looks weak mainly because of this shift.
* Logistic regression matches or beats gradient boosting, so there is no evidence of strong non-linear structure. Use the model as a risk *ranking* tool, recalibrate on recent data before applying thresholds, and find out what drives the period-level swings to improve it substantially.